In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd /content/drive/MyDrive/Colab Notebooks/CAFA6

In [ ]:
pip install torch transformers biopython obonet networkx pandas scikit-learn

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from Bio import SeqIO
import pandas as pd
from tqdm import tqdm

model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Chuyển sang GPU nếu có
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

def get_embeddings(fasta_file, max_len=1022):
    """Đọc file FASTA và trích xuất embedding từ lớp cuối cùng của ESM-2"""
    ids = []
    embeddings = []

    # Đọc file fasta bằng BioPython
    sequences = list(SeqIO.parse(fasta_file, "fasta"))

    print(f"Processing {len(sequences)} sequences...")

    # Batch processing (xử lý từng cái hoặc theo batch nhỏ để tránh tràn RAM)
    with torch.no_grad():
        for record in tqdm(sequences):
            seq = str(record.seq)[:max_len] # Cắt chuỗi nếu quá dài

            # Tokenize
            inputs = tokenizer(seq, return_tensors="pt", padding=False, truncation=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Forward pass
            outputs = model(**inputs)

            # Lấy vector đại diện (CLS token hoặc Mean pooling).
            # Ở đây dùng Mean Pooling (trung bình cộng các token) cho đơn giản
            emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

            ids.append(record.id)
            embeddings.append(emb[0]) # Flatten

    return ids, embeddings

train_ids, train_embeddings = get_embeddings("/content/drive/MyDrive/Colab Notebooks/CAFA6/data/Train/train_sequences.fasta")

In [ ]:
import numpy as np

# Load labels
terms_df = pd.read_csv("train_terms.tsv", sep="\t")

# Chọn Top 500 terms phổ biến nhất
top_terms = terms_df['term'].value_counts().index[:500]
term_to_idx = {term: i for i, term in enumerate(top_terms)}

def get_labels(protein_ids, terms_df, term_to_idx):
    num_classes = len(term_to_idx)
    labels = np.zeros((len(protein_ids), num_classes))

    # Tạo dictionary để tra cứu nhanh: protein_id -> set of terms
    id_to_terms = terms_df.groupby('EntryID')['term'].apply(set).to_dict()

    for i, pid in enumerate(protein_ids):
        if pid in id_to_terms:
            protein_terms = id_to_terms[pid]
            for term in protein_terms:
                if term in term_to_idx:
                    labels[i, term_to_idx[term]] = 1.0
    return labels

In [ ]:
import torch.nn as nn

class ProteinGoClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(ProteinGoClassifier, self).__init__()
        self.layer1 = nn.Linear(input_dim, 512)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) # Chống Overfitting
        self.layer2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        return x # Trả về logits (chưa qua Sigmoid)

# Config model
input_dim = 320 # ESM-2 small model output dimension (check lại model config nếu dùng bản lớn hơn)
num_classes = 500
model_classifier = ProteinGoClassifier(input_dim, num_classes).to(device)

# Loss function: Dùng BCEWithLogitsLoss cho bài toán Multi-label
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_classifier.parameters(), lr=1e-3)